# EXP010_MECHANISMS_V1 — frozen-model response study
A100 inference only. No training, detector selection, budget relaxation, final calibration or NASA TEST evaluation.

Run All → authorise Drive → upload the supplied bundle once. One active notebook only. This notebook and new GPU workflow have not been executed on Colab during preparation. CPU mechanism and transactional tests passed. Exact historical nominal replay remains mandatory.


In [ ]:
import os,sys,json,hashlib,subprocess,shutil,time,zipfile,platform
from pathlib import Path,PurePosixPath
from google.colab import drive,files
subprocess.run(['nvidia-smi','--query-gpu=name,driver_version','--format=csv'],check=True)
drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive/NASA_GCN_EXP010_MECHANISMS_V1');BASE.mkdir(parents=True,exist_ok=True)
OUT=BASE/'results';OUT.mkdir(exist_ok=True)
lease=OUT/'ACTIVE_RUN.json'
if lease.exists()and time.time()-json.loads(lease.read_text())['heartbeat']<600:raise RuntimeError('A run may still be active. Stop it and wait ten minutes after its last heartbeat.')
BUNDLE=BASE/'EXP010_MECHANISMS_V1_bundle.zip';EXPECTED='d61cd07a6e5c7086748a6eb82affa111ba8a20cdcbcc20015cb1e5b8810696dc'
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
if not BUNDLE.exists():
    uploaded=files.upload();matches=[b for b in uploaded.values()if hashlib.sha256(b).hexdigest()==EXPECTED]
    assert len(matches)==1,'Upload the supplied EXP010 bundle.'
    tmp=BUNDLE.with_suffix('.uploading');tmp.write_bytes(matches[0]);os.replace(tmp,BUNDLE)
assert sha(BUNDLE)==EXPECTED
WORK=Path('/content/exp010_mechanisms_v1');ROOT=WORK/'package';ROOT.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(BUNDLE)as z:
    assert len(z.namelist())==len(set(z.namelist()))
    for item in z.infolist():
        q=PurePosixPath(item.filename);assert not q.is_absolute()and '..'not in q.parts and (item.external_attr>>16)&0o170000!=0o120000
    z.extractall(ROOT)
m=json.loads((ROOT/'MANIFEST.json').read_text())
for f,h in m.items():assert sha(ROOT/f)==h,f
parent=json.loads((ROOT/'provenance/RUN_FREEZE.json').read_text())
assert platform.python_version()==parent['environment']['python'],'Python differs from the governing GPU environment. Stop; no silent portability change.'
print('Focused input bundle verified. Final calibration and NASA TEST/labels are absent.')


In [ ]:
LOGS=OUT/'logs';LOGS.mkdir(exist_ok=True);STAMP=time.strftime('%Y%m%dT%H%M%SZ',time.gmtime())
setup_log=LOGS/f'setup_{STAMP}.log'
def logged(command,env=None):
    with setup_log.open('a',buffering=1)as log:
        log.write('COMMAND '+repr(command)+'\n')
        process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
        try:
            for line in process.stdout:print(line,end='');log.write(line)
            rc=process.wait()
        except BaseException:
            process.terminate()
            try:process.wait(timeout=30)
            except subprocess.TimeoutExpired:process.kill();process.wait()
            raise
        assert rc==0,'Provisioning/check failed; preserve setup log.'
ENV=WORK/'venv';PY=ENV/'bin/python';READY=ENV/'EXP010_READY.json';LOCK=ROOT/'provenance/requirements-resolved.txt'
if not PY.exists():
    logged([sys.executable,'-m','pip','install','--disable-pip-version-check','virtualenv'])
    logged([sys.executable,'-m','virtualenv',str(ENV)])
if not READY.exists():logged([str(PY),'-m','pip','install','--disable-pip-version-check','-r',str(LOCK),'--extra-index-url','https://download.pytorch.org/whl/cu130'])
logged([str(PY),'-m','pip','check'])
resolved=subprocess.check_output([str(PY),'-m','pip','freeze','--all'],text=True)
if sorted(resolved.splitlines())!=sorted(LOCK.read_text().splitlines()):
    (OUT/'ACTUAL_REQUIREMENTS_MISMATCH.txt').write_text(resolved);raise RuntimeError('Pinned environment mismatch. Do not proceed.')
READY.write_text(json.dumps({'requirements_sha256':sha(LOCK)}));(OUT/'requirements-resolved.txt').write_text(resolved)
RUN_ENV=os.environ.copy();RUN_ENV.update(PYTHONHASHSEED='17',CUBLAS_WORKSPACE_CONFIG=':4096:8',OMP_NUM_THREADS='1',MKL_NUM_THREADS='1',OPENBLAS_NUM_THREADS='1',PYTHONUNBUFFERED='1',PYTHONDONTWRITEBYTECODE='1')
for script in ['test_mechanisms.py','test_trace_io.py','test_exp008.py']:logged([str(PY),str(ROOT/'src'/script)],env=RUN_ENV)


In [ ]:
cmd=[str(PY),'-u',str(ROOT/'src/run_exp010.py'),'--output',str(OUT)]
with (LOGS/f'run_{STAMP}.log').open('a',buffering=1)as log:
    process=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=RUN_ENV)
    try:
        for line in process.stdout:print(line,end='');log.write(line)
        rc=process.wait()
    except BaseException:
        process.terminate()
        try:process.wait(timeout=30)
        except subprocess.TimeoutExpired:process.kill();process.wait()
        raise
assert rc==0,'Stopped safely. Preserve committed units and inspect the log.'
subprocess.run([str(PY),str(ROOT/'src/verify_exp010.py'),'--output',str(OUT)],check=True,env=RUN_ENV)
print((OUT/'STUDY_CLOSE.json').read_text())


In [ ]:
EXPORT=BASE/'exports';EXPORT.mkdir(exist_ok=True)
for full,name in [(False,'EXP010_MECHANISMS_V1_summary.zip'),(True,'EXP010_MECHANISMS_V1_results.zip')]:
    final=EXPORT/name;tmp=final.with_suffix('.tmp')
    with zipfile.ZipFile(tmp,'w',zipfile.ZIP_DEFLATED)as z:
        for f in sorted(OUT.rglob('*')):
            if f.is_file()and f.name!='ACTIVE_RUN.json'and '.tmp-'not in f.name and (full or 'units'not in f.relative_to(OUT).parts):z.write(f,str(f.relative_to(OUT)))
    os.replace(tmp,final);final.with_suffix('.sha256').write_text(sha(final)+'  '+final.name+'\n');print(final)
print('STOP: diagnostic study only. Send both exports and the executed notebook for scientific audit. No detector was selected.')
files.download(str(EXPORT/'EXP010_MECHANISMS_V1_summary.zip'))
